<a href="https://colab.research.google.com/github/sawicka-byte/GWB43-groundwater-drought-DIPI/blob/main/01_preprocessing_SPI_SGI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 Preprocessing of SPI and SGI

This notebook performs the preprocessing stage of the reproducible workflow.

It loads monthly groundwater-level and precipitation data, harmonizes date fields, merges metadata, and exports processed intermediate SPI and SGI datasets for the subsequent analytical steps.

## Inputs and outputs

### Input files
- `data_input/groundwater_levels_monthly.csv`
- `data_input/metadata_piezometers.csv`
- `data_input/precipitation_monthly.csv`

### Output files
- `data_processed/SPI_monthly.csv`
- `data_processed/SGI_monthly.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
REPO_NAME = "GWB43-groundwater-drought-DIPI"
REPO_URL = "https://github.com/sawicka-byte/GWB43-groundwater-drought-DIPI.git"

repo_path = Path(f"/content/{REPO_NAME}")

if not repo_path.exists():
    !git clone {REPO_URL}

repo_root = repo_path

print("Repository root:", repo_root)
print("Repository exists:", repo_root.exists())

Cloning into 'GWB43-groundwater-drought-DIPI'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 213 (delta 29), reused 1 (delta 1), pack-reused 154 (from 1)
Receiving objects: 100% (213/213), 269.63 KiB | 5.50 MiB/s, done.
Resolving deltas: 100% (98/98), done.
Repository root: /content/GWB43-groundwater-drought-DIPI
Repository exists: True


In [3]:
import os
from pathlib import Path

REPO_NAME = "GWB43-groundwater-drought-DIPI"
REPO_URL = "https://github.com/sawicka-byte/GWB43-groundwater-drought-DIPI.git"

if not Path(f"/content/{REPO_NAME}").exists():
    !git clone {REPO_URL}

repo_root = Path(f"/content/{REPO_NAME}")
print("Repository root:", repo_root)
print("Repository exists:", repo_root.exists())

Repository root: /content/GWB43-groundwater-drought-DIPI
Repository exists: True


In [4]:
input_dir = repo_root / "data_input"
output_dir = repo_root / "data_processed"

groundwater_file = input_dir / "groundwater_levels_monthly.csv"
metadata_file = input_dir / "metadata_piezometers.csv"
precip_file = input_dir / "precipitation_monthly.csv"

print("Input directory:", input_dir)
print("Output directory:", output_dir)

Input directory: /content/GWB43-groundwater-drought-DIPI/data_input
Output directory: /content/GWB43-groundwater-drought-DIPI/data_processed


In [5]:
gw = pd.read_csv(groundwater_file)
meta = pd.read_csv(metadata_file)
prec = pd.read_csv(precip_file)

print("Groundwater data shape:", gw.shape)
print("Metadata shape:", meta.shape)
print("Precipitation data shape:", prec.shape)

Groundwater data shape: (4082, 4)
Metadata shape: (16, 10)
Precipitation data shape: (460, 6)


In [6]:
display(gw.head())
display(meta.head())
display(prec.head())

,date,piezometer_id,groundwater_level_m,cluster_id
0,2004-07-01,II/1271/1,97.26,1
1,2004-08-01,II/1271/1,97.01,1
2,2004-09-01,II/1271/1,96.86,1
3,2004-10-01,II/1271/1,96.83,1
4,2004-11-01,II/1271/1,96.92,1


,cluster_id,aquifer_group,piezometer_id,x_coord_2180,y_coord_2180,top_depth_m,bottom_depth_m,well_depth_m,stratigraphy,lithology
0,1,shallow_unconfined,II/1271/1,441727.379465,523964.377980,4.05,12.1,28.0,Q,sand
1,1,shallow_unconfined,II/1272/1,406124.320712,559613.681590,3.00,4.6,5.5,Q,sand
2,1,shallow_unconfined,II/1273/1,457116.257396,519137.508988,1.86,19.0,19.0,Q,sand
3,1,shallow_unconfined,II/1274/1,437254.525319,574337.265338,4.36,23.0,23.0,Q,sand
4,1,shallow_unconfined,II/1274/2,437254.525319,574337.265338,4.36,23.0,23.0,Q,sand


,date,imgw_Zalesie,imgw_Trzemeszno,imgw_Pakość,imgw_Jaksice,imgw_Gębice
0,1986-01-01,86.8,30.4,45.6,55.5,44.2
1,1986-02-01,14.2,15.1,8.9,7.3,11.4
2,1986-03-01,17.6,23.8,22.1,22.1,18.2
3,1986-04-01,22.6,37.3,34.2,19.2,32.8
4,1986-05-01,83.8,61.4,45.1,39.9,74.1


## Preprocessing notes

This step should standardize date fields, verify column names, inspect missing values, and prepare monthly time series in a consistent tabular format.

If required, file-specific column names can be harmonized in the cleaning section below.

In [7]:
# Parse dates
gw["date"] = pd.to_datetime(gw["date"])
prec["date"] = pd.to_datetime(prec["date"])

# Sort values
gw = gw.sort_values(["piezometer_id", "date"]).reset_index(drop=True)
prec = prec.sort_values("date").reset_index(drop=True)

# Merge metadata
gw_clean = gw.merge(
    meta[["piezometer_id", "aquifer_group"]],
    on="piezometer_id",
    how="left"
)

print("Groundwater cleaned shape:", gw_clean.shape)
display(gw_clean.head())

Groundwater cleaned shape: (4082, 5)


,date,piezometer_id,groundwater_level_m,cluster_id,aquifer_group
0,1994-04-01,II/1065/1,76.79,3,deep_confined
1,1994-05-01,II/1065/1,76.83,3,deep_confined
2,1994-06-01,II/1065/1,76.81,3,deep_confined
3,1994-07-01,II/1065/1,77.01,3,deep_confined
4,1994-08-01,II/1065/1,76.97,3,deep_confined


In [8]:
# ------------------------------------------------------------------
# SPI preprocessing
# Monthly precipitation table prepared for SPI calculation
# ------------------------------------------------------------------

spi_monthly = prec.copy()

# Optional: mean precipitation across all IMGW stations
station_cols = [col for col in spi_monthly.columns if col != "date"]
spi_monthly["precip_mean_all_stations"] = spi_monthly[station_cols].mean(axis=1)

print("SPI monthly table prepared.")
print("Stations:", station_cols)
display(spi_monthly.head())

SPI monthly table prepared.
Stations: ['imgw_Zalesie', 'imgw_Trzemeszno', 'imgw_Pakość', 'imgw_Jaksice', 'imgw_Gębice']


,date,imgw_Zalesie,imgw_Trzemeszno,imgw_Pakość,imgw_Jaksice,imgw_Gębice,precip_mean_all_stations
0,1986-01-01,86.8,30.4,45.6,55.5,44.2,52.50
1,1986-02-01,14.2,15.1,8.9,7.3,11.4,11.38
2,1986-03-01,17.6,23.8,22.1,22.1,18.2,20.76
3,1986-04-01,22.6,37.3,34.2,19.2,32.8,29.22
4,1986-05-01,83.8,61.4,45.1,39.9,74.1,60.86


In [9]:
# ------------------------------------------------------------------
# SGI preprocessing
# Monthly groundwater table prepared for SGI calculation
# ------------------------------------------------------------------

sgi_monthly = gw_clean.copy()

# Optional cluster monthly mean
cluster_monthly_mean = (
    sgi_monthly
    .groupby(["date", "cluster_id"], as_index=False)["groundwater_level_m"]
    .mean()
    .rename(columns={"groundwater_level_m": "cluster_mean_gwl_m"})
)

sgi_monthly = sgi_monthly.merge(
    cluster_monthly_mean,
    on=["date", "cluster_id"],
    how="left"
)

print("SGI monthly table prepared.")
display(sgi_monthly.head())

SGI monthly table prepared.


,date,piezometer_id,groundwater_level_m,cluster_id,aquifer_group,cluster_mean_gwl_m
0,1994-04-01,II/1065/1,76.79,3,deep_confined,81.760
1,1994-05-01,II/1065/1,76.83,3,deep_confined,81.800
2,1994-06-01,II/1065/1,76.81,3,deep_confined,81.765
3,1994-07-01,II/1065/1,77.01,3,deep_confined,81.845
4,1994-08-01,II/1065/1,76.97,3,deep_confined,81.865


In [10]:
output_dir.mkdir(parents=True, exist_ok=True)

spi_outfile = output_dir / "SPI_monthly.csv"
sgi_outfile = output_dir / "SGI_monthly.csv"

spi_monthly.to_csv(spi_outfile, index=False)
sgi_monthly.to_csv(sgi_outfile, index=False)

print("Saved:")
print("-", spi_outfile)
print("-", sgi_outfile)

Saved:
- /content/GWB43-groundwater-drought-DIPI/data_processed/SPI_monthly.csv
- /content/GWB43-groundwater-drought-DIPI/data_processed/SGI_monthly.csv


## Reproducibility note

This notebook represents the preprocessing stage of the workflow. The exported SPI and SGI tables serve as intermediate analytical products for subsequent event-metric and DIPI-construction steps.